In [2]:
!pip -q install -U \
langchain \
langchain-core \
langchain-community \
langgraph \
langchain-google-genai \
langchain-groq \
chromadb \
pymupdf \
pypdf \
pandas \
python-dotenv \
tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.8/247.8 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 110.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [3]:
import os
import fitz
import pandas as pd

from google.colab import drive

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import GoogleGenerativeAIEmbeddings

from langchain_community.vectorstores import Chroma

/tmp/ipykernel_1257/717302095.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [10]:
import os

print(os.path.exists("/content/drive"))
print(os.listdir("/content"))

True
['.config', 'drive', 'sample_data']


In [11]:
from google.colab import drive

drive.flush_and_unmount()

print("Drive unmounted.")

Drive not mounted, so nothing to flush and unmount.
Drive unmounted.


In [12]:
!rm -rf /content/drive

In [13]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [14]:
BASE = "/content/drive/MyDrive/Insurance-Agentic-AI"

folders = [
    BASE,
    BASE + "/data",
    BASE + "/data/policies",
    BASE + "/vector_db"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully.")

Project folders created successfully.


In [15]:
os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

In [16]:
PDF_FOLDER = BASE + "/data/policies"

documents = []

for file in os.listdir(PDF_FOLDER):

    if file.lower().endswith(".pdf"):

        pdf_path = os.path.join(PDF_FOLDER, file)

        pdf = fitz.open(pdf_path)

        text = ""

        for page in pdf:
            text += page.get_text()

        documents.append(
            Document(
                page_content=text,
                metadata={"source": file}
            )
        )

print("Total PDFs :", len(documents))

Total PDFs : 3


In [17]:
import os

PDF_FOLDER = "/content/drive/MyDrive/Insurance-Agentic-AI/data/policies"

print("Folder exists:", os.path.exists(PDF_FOLDER))

if os.path.exists(PDF_FOLDER):
    files = os.listdir(PDF_FOLDER)
    print("Files:", files)
    print("PDF files:", [f for f in files if f.lower().endswith(".pdf")])

Folder exists: True
Files: ['Life Insurance.pdf', 'Health Insurance.pdf', 'Home Insurance.pdf']
PDF files: ['Life Insurance.pdf', 'Health Insurance.pdf', 'Home Insurance.pdf']


In [18]:
print("Number of documents:", len(documents))

Number of documents: 3


In [20]:
splitter = RecursiveCharacterTextSplitter(

    chunk_size=1000,

    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print("Total Chunks :", len(chunks))

Found 3 PDF(s).
✅ Loaded: Life Insurance.pdf
✅ Loaded: Health Insurance.pdf
✅ Loaded: Home Insurance.pdf

Total documents loaded: 3
Found 3 PDF(s).
✅ Loaded: Life Insurance.pdf
✅ Loaded: Health Insurance.pdf
✅ Loaded: Home Insurance.pdf

Total documents loaded: 3


In [21]:
splitter = RecursiveCharacterTextSplitter(

    chunk_size=1000,

    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print("Total Chunks :", len(chunks))

Total Chunks : 1425


In [31]:
!pip install -q -U langchain-huggingface sentence-transformers chromadb
from langchain_huggingface import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [34]:
vector_db = Chroma.from_documents(

    documents=chunks,

    embedding=embedding,

    persist_directory=BASE + "/vector_db"
)

In [35]:
vector_db.persist()

print("Vector Database Created Successfully")

Vector Database Created Successfully


/tmp/ipykernel_1257/2046289704.py:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_db.persist()


In [36]:
db = Chroma(

    persist_directory=BASE + "/vector_db",

    embedding_function=embedding
)

/tmp/ipykernel_1257/257739704.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(


In [37]:
query = "What are the waiting periods for health insurance?"

docs = retriever.invoke(query)

for doc in docs:

    print("="*80)
    print(doc.metadata)
    print(doc.page_content[:800])

NameError: name 'retriever' is not defined